In [1]:
import pandas as pd
import numpy as np
import time
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.utils import resample
from sklearn import preprocessing
from warnings import simplefilter
from imblearn.under_sampling import RandomUnderSampler

# Suppress FutureWarning messages
simplefilter(action='ignore', category=FutureWarning)

# Record start time
start_time = time.time()

In [3]:
import pandas as pd
import numpy as np
from sklearn import preprocessing
from sklearn.preprocessing import StandardScaler

# Define the list of file paths (without .csv extension)
all_files = [
    "dataset/All_BestFirst_test", "dataset/All_BestFirst", "dataset/All_Infogain_test", 
    "dataset/All_Infogain", "dataset/Malware_BestFirst", "dataset/Malware_Infogain_test", 
    "dataset/Malware_Infogain", "dataset/output2", "dataset/output3", "dataset/Phishing_BestFirst", 
    "dataset/Phishing_Infogain", "dataset/Phishing", "dataset/Scenario-A-merged_5s", 
    "dataset/Scenario-B-merged_5s", "dataset/Spam_BestFirst_test", "dataset/Spam_Infogain_test", 
    "dataset/Spam_Infogain", "dataset/Spam"
]

# Initialize a list to store processed DataFrames
processed_dataframes = []

# Create a StandardScaler instance for normalization
std_scaler = StandardScaler()

# Function for normalization
def normalize_dataframe(df, columns_to_normalize):
    df[columns_to_normalize] = std_scaler.fit_transform(df[columns_to_normalize])
    return df

for file_path in all_files:
    # Read CSV file
    df = pd.read_csv(file_path + ".csv", encoding='iso-8859-2', engine='python')
    df.columns = df.columns.str.strip()  # Remove any whitespace from column names
    
    # Check if "Flow Duration" exists in the columns before proceeding
    if "Flow Duration" in df.columns:
        # Drop rows with missing Flow Duration values
        df = df.drop(df[pd.isnull(df["Flow Duration"])].index)
    else:
        print(f"'Flow Duration' column not found in {file_path}. Skipping missing value check.")
    
    # Replace infinite values with NaN
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    
    # Drop rows with NaN values
    df.dropna(inplace=True)
    
    # Normalize numeric columns
    numeric_columns = df.select_dtypes(include='number').columns
    df[numeric_columns] = df[numeric_columns].astype(np.float32)
    df = normalize_dataframe(df.copy(), numeric_columns)
    
    # Identify and handle categorical columns
    string_columns = [col for col in df.columns if df[col].dtype == "object"]
    try:
        string_columns.remove('Label')  # Adjusted to 'Label' without leading space
    except ValueError:
        pass
    
    # Convert categorical columns to numeric
    label_encoder_X = preprocessing.LabelEncoder()
    for col in string_columns:
        try:
            df[col] = label_encoder_X.fit_transform(df[col])
        except:
            df[col] = df[col].replace('Infinity', -1)
   
    # Append the processed DataFrame to the list
    processed_dataframes.append(df)
    print("Preprocessing and undersampling of file", file_path, " is done")

# Concatenate the processed DataFrames
combined_dataframe = pd.concat(processed_dataframes, ignore_index=True)

# Save the concatenated DataFrame to a new CSV file
combined_dataframe.to_csv("combined_data.csv", index=False)
print("Concatenation and saving to CSV is done")

'Flow Duration' column not found in dataset/All_BestFirst_test. Skipping missing value check.
Preprocessing and undersampling of file dataset/All_BestFirst_test  is done
'Flow Duration' column not found in dataset/All_BestFirst. Skipping missing value check.
Preprocessing and undersampling of file dataset/All_BestFirst  is done
'Flow Duration' column not found in dataset/All_Infogain_test. Skipping missing value check.
Preprocessing and undersampling of file dataset/All_Infogain_test  is done
'Flow Duration' column not found in dataset/All_Infogain. Skipping missing value check.
Preprocessing and undersampling of file dataset/All_Infogain  is done
'Flow Duration' column not found in dataset/Malware_BestFirst. Skipping missing value check.
Preprocessing and undersampling of file dataset/Malware_BestFirst  is done
'Flow Duration' column not found in dataset/Malware_Infogain_test. Skipping missing value check.
Preprocessing and undersampling of file dataset/Malware_Infogain_test  is done


ParserError: ',' expected after '"'